In [1]:
# ================================================
# 1. IMPORTS
# ================================================
import kagglehub
import os
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications import VGG19, ResNet50
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

# ================================================
# 2. DOWNLOAD DATASET USING KAGGLEHUB
# ================================================
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print("Dataset path:", path)

# ================================================
# 3. DATA LOADING (TRAINING + TESTING)
# ================================================
CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]

DATA_DIRS = [
    os.path.join(path, "Training"),
    os.path.join(path, "Testing")
]

img_size = 224
X = []
y = []

print("\nLoading images (may take time)...\n")

for data_dir in DATA_DIRS:
    for label, cls in enumerate(CLASSES):
        folder = os.path.join(data_dir, cls)

        if not os.path.isdir(folder):
            print("Skipping missing folder:", folder)
            continue

        print("Reading:", folder)

        for file in os.listdir(folder):
            img_path = os.path.join(folder, file)

            try:
                img = load_img(img_path, target_size=(img_size, img_size))
                img = img_to_array(img) / 255.0   # normalize
                X.append(img)
                y.append(label)
            except:
                continue

X = np.array(X)
y = np.array(y)

print("\nTotal Images Loaded:", len(X))

# ================================================
# 4. TRAIN–TEST SPLIT (80/20)
# ================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, " Test:", X_test.shape)

# ================================================
# 5. ENSEMBLE MODEL (VGG19 + ResNet50)
# ================================================
input_tensor = Input(shape=(224, 224, 3))

# --- VGG19 ---
vgg = VGG19(include_top=False, weights="imagenet", input_tensor=input_tensor)
vgg.trainable = False
vgg_out = GlobalAveragePooling2D()(vgg.output)

# --- ResNet50 ---
res = ResNet50(include_top=False, weights="imagenet", input_tensor=input_tensor)
res.trainable = False
res_out = GlobalAveragePooling2D()(res.output)

# --- Concatenate ---
merged = Concatenate()([vgg_out, res_out])

# --- Full neural layers ---
x = Dense(512, activation="relu")(merged)
x = Dropout(0.25)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.25)(x)
output = Dense(4, activation="softmax")(x)

model = Model(inputs=input_tensor, outputs=output)

# ================================================
# 6. COMPILE
# ================================================
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ================================================
# 7. TRAINING
# ================================================
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=15,
    batch_size=32,
    shuffle=True
)

# ================================================
# 8. EVALUATION
# ================================================
loss, acc = model.evaluate(X_test, y_test)
print("\nFINAL TEST ACCURACY:", acc)


C:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset path: C:\Users\HP\.cache\kagglehub\datasets\masoudnickparvar\brain-tumor-mri-dataset\versions\1

Loading images (may take time)...

Reading: C:\Users\HP\.cache\kagglehub\datasets\masoudnickparvar\brain-tumor-mri-dataset\versions\1\Training\glioma
Reading: C:\Users\HP\.cache\kagglehub\datasets\masoudnickparvar\brain-tumor-mri-dataset\versions\1\Training\meningioma
Reading: C:\Users\HP\.cache\kagglehub\datasets\masoudnickparvar\brain-tumor-mri-dataset\versions\1\Training\notumor
Reading: C:\Users\HP\.cache\kagglehub\datasets\masoudnickparvar\brain-tumor-mri-dataset\versions\1\Training\pituitary
Reading: C:\Users\HP\.cache\kagglehub\datasets\masoudnickparvar\brain-tumor-mri-dataset\versions\1\Testing\glioma
Reading: C:\Users\HP\.cache\kagglehub\datasets\masoudnickparvar\brain-tumor-mri-dataset\versions\1\Testing\meningioma
Reading: C:\Users\HP\.cache\kagglehub\datasets\masoudnickparvar\brain-tumor-mri-dataset\versions\1\Testing\notumor
Reading: C:\Users\HP\.cache\kagglehub\dataset

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_pad (ZeroPadding2D)     │ (None, 230, 230, 3)       │               0 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_conv (Conv2D)           │ (None, 112, 112, 64)      │           9,472 │ conv1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_bn (BatchNormalization) │ (None, 112, 112, 64)      │             256 │ conv1_conv[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_relu (Activation)       │ (None, 112, 112, 64)      │               0 │ conv1_bn[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pad (ZeroPadding2D)     │ (None, 114, 114, 64)      │               0 │ conv1_relu[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pool (MaxPooling2D)     │ (None, 56, 56, 64)        │               0 │ pool1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_conv (Conv2D)  │ (None, 56, 56, 64)        │           4,160 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_1_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_1_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_conv (Conv2D)  │ (None, 56, 56, 64)        │          36,928 │ conv2_block1_1_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_bn             │ (None, 56, 56, 64)        │             256 │ conv2_block1_2_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_relu           │ (None, 56, 56, 64)        │               0 │ conv2_block1_2_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_0_conv (Conv2D)  │ (None, 56, 56, 256)       │          16,640 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_3_conv (Conv2D)  │ (None, 56, 56, 256)       │          16,640 │ conv2_block1_2_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼───────────────

 Total params: 45,055,684 (171.87 MB)

 Trainable params: 1,443,588 (5.51 MB)

 Non-trainable params: 43,612,096 (166.37 MB)

Epoch 1/15
176/176 ━━━━━━━━━━━━━━━━━━━━ 1173s 7s/step - accuracy: 0.4457 - loss: 1.2114 - val_accuracy: 0.6790 - val_loss: 0.9057
Epoch 2/15
176/176 ━━━━━━━━━━━━━━━━━━━━ 1159s 7s/step - accuracy: 0.6493 - loss: 0.8735 - val_accuracy: 0.7537 - val_loss: 0.6906
Epoch 3/15
176/176 ━━━━━━━━━━━━━━━━━━━━ 1156s 7s/step - accuracy: 0.7296 - loss: 0.7048 - val_accuracy: 0.7865 - val_loss: 0.5778
Epoch 4/15
176/176 ━━━━━━━━━━━━━━━━━━━━ 1097s 6s/step - accuracy: 0.7679 - loss: 0.5996 - val_accuracy: 0.8064 - val_loss: 0.5000
Epoch 5/15
176/176 ━━━━━━━━━━━━━━━━━━━━ 1089s 6s/step - accuracy: 0.7827 - loss: 0.5504 - val_accuracy: 0.7986 - val_loss: 0.4963
Epoch 6/15
176/176 ━━━━━━━━━━━━━━━━━━━━ 1074s 6s/step - accuracy: 0.8095 - loss: 0.4951 - val_accuracy: 0.8399 - val_loss: 0.4200
Epoch 7/15
176/176 ━━━━━━━━━━━━━━━━━━━━ 1082s 6s/step - accuracy: 0.8179 - loss: 0.4697 - val_accuracy: 0.8577 - val_loss: 0.3920
Epoch 8/15
176/176 ━━━━━━━━━━━━━━━━━━━━ 1122s 6s/step - accuracy: 0.8220 - loss: 0.4527 - 